## Average bond lifetime + bonding rate from averaging lifetime

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

f = "data/patches/0.7prodpatch_chainlength_30_patchspacing_5_N_beads_24000_gaussA_60_r0_0.25_gaussB_10_seed_12345.bin.txt"

N_patches = 4000
frame = 11

with open(f, "r") as file:
    for i in range((frame - 1) * (9 + N_patches) + 1):
        line = file.readline()
    # for i in range(9):
    print("Timestep:", file.readline())

df = pd.read_csv(f, skiprows=frame*9+(frame - 1) * N_patches, sep=' ', nrows=N_patches, header=None, names=["id", "mol", "x", "y", "z", "patch_coord", "patch_cluster"])
df.sort_values(by=["patch_coord", "id"], ascending=True, inplace=True)
print(df[df["patch_coord"] == 2])
print(df[df["patch_cluster"] == 8575])

In [ ]:
patch_cluster_list = np.unique(df["patch_cluster"].values)
print(len(patch_cluster_list))
cluster_size_list = np.zeros(len(patch_cluster_list))
print(len(cluster_size_list))


for i, patch_cluster in enumerate(patch_cluster_list):
    this_patch = df[df["patch_cluster"] == patch_cluster]
    # print("Patch id:", patch_cluster)
    # print(this_patch)
    cluster_size_list[i] = np.max(this_patch[["patch_coord"]].values)
    # print(f"Cluster {patch_cluster} has size {cluster_size_list[i]}")


# percentage of bonds with 3 patches
num_3_patch_bonds = np.sum(cluster_size_list == 2)
print(num_3_patch_bonds)
total_bonds = np.sum(cluster_size_list == 2) + np.sum(cluster_size_list == 1) # only consider bonds with at least 1 patch
print(total_bonds)
# percentage_3_patch_bonds = (num_3_patch_bonds / total_bonds) * 100
# print(f"Percentage of bonds with 3 patches: {percentage_3_patch_bonds:.2f}%")

# histogram of patch_coord
plt.figure(figsize=(8, 6))
plt.hist(df["patch_coord"], bins=np.arange(-0.5, 4.5, 1), density=True)
plt.title("Distribution of Patch Coordination Numbers")
plt.xlabel("Patch Coordination Number")
plt.ylabel("Probability Density")
plt.grid()
plt.show()  

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


for A in [80, 90, 100, 110, 120, 130]:
    two_patch_bond_list = []
    three_patch_bond_list = []
    for cutoff in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
        filename = f"data/patches/{cutoff}prodpatch_chainlength_31_patchspacing_5_N_beads_24800_gaussA_{A}_r0_0.25_gaussB_10_seed_12345.bin.txt"
        N_patches = 4000
        frame = 11

        # with open(filename, "r") as file:
        #     for i in range((frame - 1) * (9 + N_patches) + 1):
        #         line = file.readline()
        #     print("Timestep:", file.readline())

        df = pd.read_csv(filename, skiprows=frame*9+(frame - 1) * N_patches, sep=' ', nrows=N_patches, header=None, names=["id", "mol", "x", "y", "z", "patch_coord", "patch_cluster"])
        df.sort_values(by=["patch_coord", "id"], ascending=True, inplace=True)

        patch_cluster_list = np.unique(df["patch_cluster"].values)
        cluster_size_list = np.zeros(len(patch_cluster_list))

        for i, patch_cluster in enumerate(patch_cluster_list):
            this_patch = df[df["patch_cluster"] == patch_cluster]
            cluster_size_list[i] = np.max(this_patch[["patch_coord"]].values) + 1

        two_patch_bond_list.append(np.sum(cluster_size_list == 2)/2)
        three_patch_bond_list.append(np.sum(cluster_size_list == 3)/3)
    
    figure = plt.figure(figsize=(8, 6))
    plt.plot([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7], two_patch_bond_list, marker='o', label="2-patch bonds")
    plt.plot([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7], three_patch_bond_list, marker='o', label="3-patch bonds")
    plt.title(f"Number of Bonds vs Cutoff for A={A}")
    plt.xlabel("Cutoff")
    plt.ylabel("Number of Bonds")
    plt.grid()
    plt.legend()
    plt.show()

## Offtime algorithm with different $r_{\text{on}}, r_{\text{off}}$

With different radii for forming/breaking a bond, we should avoid jitter close to one cutoff that result in tiny lifetimes. This does require generating and reading two clustering dumps, but we can dump both in one file so we at least won't have too many files.


In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import pandas as pd

# """
# Process offtimes from clustering data file. Each frame of the file contains the following sections:
# - "ITEM: TIMESTEP" followed by the timestep on the next line
# - "ITEM: NUMBER OF ATOMS" followed by the number of entries on the next line
# - "ITEM: ATOMS" followed by the data lines for each entry, with 7 values per line: id, mol, x, y, z, patch_coord, patch_cluster

# The unbonded_patches list is a dictionary where the key is a patch's id and the value is the last timestep when a bond it was part of breaks.

# The bond_offtimes list is a list of tuples (id, start_time, end_time, offtime) for each bonded pair that has formed.

# If there are any bonded_pairs that have not created by the end of the dump, we ignore them and do not record a offtime for them.

# At the start of the simulation, all patches are unbonded, so we do not start any timers for them. Therefore, it takes at least one creation and breaking event 
# for bond offtimes to start being recorded, might take some time to get to a steady state.

# All times for breaking/creating are considered on the timestep we note them, so we should on average get the right time 
# (i.e. both events are noted AFTER they happen)

# Args:
#     - filename: path to the clustering data file
# Returns:
#     - bond_offtimes: list of tuples (id, start_time, end_time, offtime) for each bonded pair
# """
# def process_buffered_offtimes(filename_on, filename_off, timestep=0.002):
#     # init empty bond offtime list
#     bond_offtimes = []
#     # init empty bonded pair dict. lookup by id. value is end_time of the previous bond patch was part of.
#     unbonded_ids = {}

#     all_ids = set()
#     # generate a list of all patch ids from the first frame of the file
#     print(f"Reading patch ids from {filename_on}")
#     with open(filename_on, 'r') as f_on:
#         while True:
#             line = f_on.readline()
#             if line.startswith("ITEM: NUMBER OF ATOMS"): # number of entries is on next line
#                 N_patches = int(f_on.readline())
#                 print(f"Number of patches: {N_patches}")
#             if line.startswith("ITEM: ATOMS"): # data starts on next line
#                 for i in range(N_patches):
#                     line = f_on.readline().strip().split()
#                     # print(f"Adding patch id: {line[0]} to all_ids set")
#                     all_ids.add(line[0]) # add id to all_ids set
#                 break

#     print(f"Total number of patches: {len(all_ids)}")

#     with open(filename_on, 'r') as f_on:
#         with open(filename_off, 'r') as f_off: 
#             time = 0
#             while True:
#                 # stop reading BOTH if either file is done

#                 # reading f_on, only creating bonds
#                 line = f_on.readline()
#                 if not line:
#                     break
#                 if line.startswith("ITEM: TIMESTEP"): # timestep is on next line
#                     time = float(f_on.readline().strip()) * timestep # convert to time units
#                     # print(f"Processing timestep: {time / timestep:.0f} (time units: {time})")
#                 elif line.startswith("ITEM: NUMBER OF ATOMS"): # number of entries is on next line
#                     N_patches = int(f_on.readline().strip())
#                 elif line.startswith("ITEM: ATOMS"): # data starts on next line
#                     current_bonds = set() # track current bonded pairs in this frame to compare with bonded_pairs dict
#                     clusters = np.zeros((N_patches,7)) # 7 values per line: id, mol, x, y, z, patch_coord, patch_cluster
#                     for i in range(N_patches):
#                         clusters[i] = np.array(f_on.readline().strip().split(), dtype=float)
#                     # filter only consider bonds with at least 1 patch
#                     clusters = clusters[clusters[:,5] > 0] # patch_coord is in column 5
#                     # sort by patch_cluster (get bonded pairs) 
#                     clusters = clusters[clusters[:,6].argsort()] # patch_cluster is in column 6

#                     for i in range(len(clusters)-1):
#                         id1, id2 = int(clusters[i][0]), int(clusters[i+1][0]) # get ids of current and next cluster
#                         if clusters[i][6] == clusters[i+1][6]: # if same patch_cluster, they are bonded
#                             current_bonds.add(int(id1)) # add to current bonds set
#                             current_bonds.add(int(id2)) # add to current bonds set

#                     # new bonds:
#                     print(f"Current bonded ids: {len(current_bonds)}, current unbonded: {len(unbonded_ids)}")
#                     for id in current_bonds:
#                         # if id < 50:
#                             # print(f"Current bonded patch id: {id}")
#                             # print(f"Unbonded ids: {unbonded_ids}")
#                             # print(str(id) in unbonded_ids.keys())
#                         if id in unbonded_ids:
#                             start_time = unbonded_ids.pop(id) # remove from unbonded_ids and get start_time
#                             # print(f"Bond for patch {id} formed at {time:.3f} after breaking at {start_time:.3f}")
#                             offtime = time - start_time
#                             # if (offtime > timestep):
#                             # add to bond_offtimes list
#                             print(f"Bond for patch {id} broke at {start_time:.3f} and formed at {time:.3f}, offtime: {offtime:.3f}")
#                             bond_offtimes.append((id, start_time, time, offtime))
                
#                 # reading f_off, only breaking bonds
#                 line = f_off.readline()
#                 if not line or time > 20 * timestep: #TODO remove this condition, just for testing to not read entire file
#                     break
#                 if line.startswith("ITEM: TIMESTEP"): # timestep is on next line
#                     time = float(f_off.readline().strip()) * timestep # convert to time units
#                     print(f"Processing timestep: {time / timestep:.0f} (time units: {time})")
#                 elif line.startswith("ITEM: NUMBER OF ATOMS"): # number of entries is on next line
#                     N_patches = int(f_off.readline().strip())
#                 elif line.startswith("ITEM: ATOMS"): # data starts on next line
#                     current_bonds = set() # track current bonded pairs in this frame to compare with bonded_pairs dict
#                     clusters = np.zeros((N_patches,7)) # 7 values per line: id, mol, x, y, z, patch_coord, patch_cluster
#                     for i in range(N_patches):
#                         clusters[i] = np.array(f_off.readline().strip().split(), dtype=float)
#                     # filter only consider bonds with at least 1 patch
#                     clusters = clusters[clusters[:,5] > 0] # patch_coord is in column 5
#                     # sort by patch_cluster (get bonded pairs) 
#                     clusters = clusters[clusters[:,6].argsort()] # patch_cluster is in column 6
#                     for i in range(len(clusters)-1):
#                         id1, id2 = int(clusters[i][0]), int(clusters[i+1][0]) # get ids of current and next cluster
#                         if clusters[i][6] == clusters[i+1][6]: # if same patch_cluster, they are bonded
#                             current_bonds.add(id1) # add to current bonds set
#                             current_bonds.add(id2) # add to current bonds set

#                     broken_bonds = all_ids - set(unbonded_ids.keys()) - set(current_bonds) # bonds that are in unbonded_ids but not in current_bonds are broken
#                     # print("all_ids:", len(all_ids), "unbonded_ids:", len(unbonded_ids), "current_bonds:", len(current_bonds), "broken_bonds:", len(broken_bonds))
#                     # print(f"Broken bonds at timestep {time / timestep:.0f} (time units: {time}): {broken_bonds}")
#                     for id in broken_bonds:
#                         #  print("bond broken for patch id:", id)
#                          unbonded_ids[int(id)] = time
                        

#                     # print(f"Current bonded pairs: {len(bonded_pairs)}, Total bond lifetimes recorded: {len(bond_lifetimes)}")
#                     # print(bond_lifetimes)
                
#     return bond_offtimes



In [30]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.notebook import tqdm

"""
Process offtimes from clustering data file. Each frame of the file contains the following sections:
- "ITEM: TIMESTEP" followed by the timestep on the next line
- "ITEM: NUMBER OF ATOMS" followed by the number of entries on the next line
- "ITEM: ATOMS" followed by the data lines for each entry, with 7 values per line: id, mol, x, y, z, patch_coord, patch_cluster

The unbonded_patches list is a dictionary where the key is a patch's id and the value is the last timestep when a bond it was part of breaks.

The bond_offtimes list is a list of tuples (id, start_time, end_time, offtime) for each bonded pair that has formed.

If there are any bonded_pairs that have not created by the end of the dump, we ignore them and do not record a offtime for them.

At the start of the simulation, all patches are unbonded, so we do not start any timers for them. Therefore, it takes at least one creation and breaking event 
for bond offtimes to start being recorded, might take some time to get to a steady state.

All times for breaking/creating are considered on the timestep we note them, so we should on average get the right time 
(i.e. both events are noted AFTER they happen)

Args:
    - filename_on: path to the clustering data file (for creating bonds)
    - filename_off: path to the clustering data file (for breaking bonds)
    - timestep: timestep size in LJ units (default is 0.002)
Returns:
    - bond_offtimes: list of tuples (id, start_time, end_time, offtime) for each patch
"""
def process_buffered_offtimes(filename_on, filename_off, timestep=0.002):
    # init empty bond offtime list
    bond_offtimes = []
    # init empty bonded pair dict. lookup by id. value is end_time of the previous bond patch was part of.
    unbonded_ids = {}

    all_ids = set()
    N_patches = 0
    time = 0
    with open(filename_on, 'r') as f_on:
        while True:
            line = f_on.readline()
            if not line:
                break
            if line.startswith("ITEM: TIMESTEP"):
                time = float(f_on.readline().strip()) * timestep
            elif line.startswith("ITEM: NUMBER OF ATOMS") and len(all_ids) == 0:
                N_patches = int(f_on.readline())
                print(f"Number of patches: {N_patches}")
            elif line.startswith("ITEM: ATOMS") and len(all_ids) == 0:
                # read next N_patches rows using pandas (fast C engine), only first column
                ids_df = pd.read_csv(f_on, sep=r'\s+', header=None, nrows=N_patches, usecols=[0], engine='c')
                all_ids.update(ids_df.iloc[:,0].astype(int).tolist())
                print(f"Read {len(all_ids)} patch ids from {filename_on}")
                # break
    total_time = time

    print(f"Total number of patches: {len(all_ids)}")

    pbar = tqdm(total=total_time, desc="Processing bond offtimes")

    with open(filename_on, 'r') as f_on:
        with open(filename_off, 'r') as f_off: 
            time = 0
            while True:
                # stop reading BOTH if either file is done

                # reading f_on, only creating bonds
                line = f_on.readline()
                if not line:
                    break
                if line.startswith("ITEM: TIMESTEP"): # timestep is on next line
                    time = float(f_on.readline().strip()) * timestep # convert to time units
                    # print(f"Processing timestep: {time / timestep:.0f} (time units: {time})")
                elif line.startswith("ITEM: ATOMS"): # data starts on next line
                    current_bonds = set() # track current bonded pairs in this frame to compare with bonded_pairs dict
                    clusters = pd.read_csv(f_on, delimiter='\s+', header=None, nrows=N_patches, names=['id', 'mol', 'x', 'y', 'z', 'patch_coord', 'patch_cluster'])
                    bonded_clusters = clusters[clusters['patch_coord'] > 0] # filter only consider bonds with at least 1 patch (patch_coord is in column 5)
                    bonded_clusters.sort_values(by='patch_cluster', inplace=True) # sort by patch_cluster (get bonded pairs) (patch_cluster is in column 6)
                    bonded_clusters.reset_index(drop=True, inplace=True)
                    for i in range(len(bonded_clusters)-1):
                        id1, id2 = int(bonded_clusters.iloc[i, 0]), int(bonded_clusters.iloc[i+1, 0]) # get ids of current and next cluster
                        if bonded_clusters.iloc[i, 6] == bonded_clusters.iloc[i+1, 6]: # if same patch_cluster, they are bonded
                            current_bonds.add(id1) # add to current bonds set
                            current_bonds.add(id2) # add to current bonds set
                    # clusters = np.zeros((N_patches,7)) # 7 values per line: id, mol, x, y, z, patch_coord, patch_cluster
                    # for i in range(N_patches):
                    #     clusters[i] = np.array(f_on.readline().strip().split(), dtype=float)
                    # # filter only consider bonds with at least 1 patch
                    # clusters = clusters[clusters[:,5] > 0] # patch_coord is in column 5
                    # # sort by patch_cluster (get bonded pairs) 
                    # clusters = clusters[clusters[:,6].argsort()] # patch_cluster is in column 6

                    # for i in range(len(clusters)-1):
                    #     id1, id2 = int(clusters[i][0]), int(clusters[i+1][0]) # get ids of current and next cluster
                    #     if clusters[i][6] == clusters[i+1][6]: # if same patch_cluster, they are bonded
                    #         current_bonds.add(int(id1)) # add to current bonds set
                    #         current_bonds.add(int(id2)) # add to current bonds set

                    # print(f"Current bonds: {len(current_bonds)}")
                    for id in current_bonds:
                        if(type(id) != int):
                            raise Exception("hey that's not right,", type(id))
                        # on the first pass, we should FINISh this with an empty bond_offtimes and unbonded_ids
                        # print(f"Current bonded patch id: {id}")
                        # print(f"Unbonded ids: {unbonded_ids.keys()}")
                        if id in unbonded_ids.keys(): # if just formed, stop their timers
                            # raise Exception("well this was meant to happen...", type(id))
                            start_time = unbonded_ids.pop(id)
                            bond_offtimes.append((id, start_time, time, time - start_time))
                    del current_bonds # free memory

                
                # reading f_off, only breaking bonds
                line = f_off.readline()
                if not line:
                    break
                if line.startswith("ITEM: TIMESTEP"): # timestep is on next line
                    time = float(f_off.readline().strip()) * timestep # convert to time units
                    # print(f"Processing timestep: {time / timestep:.0f} (time units: {time})")
                elif line.startswith("ITEM: ATOMS"): # data starts on next line
                    current_bonds = set() # track current bonded pairs in this frame to compare with bonded_pairs dict
                    clusters = pd.read_csv(f_off, delimiter='\s+', header=None, nrows=N_patches, names=['id', 'mol', 'x', 'y', 'z', 'patch_coord', 'patch_cluster'])
                    bonded_clusters = clusters[clusters['patch_coord'] > 0] # filter only consider bonds with at least 1 patch (patch_coord is in column 5)
                    bonded_clusters.sort_values(by='patch_cluster', inplace=True) # sort by patch_cluster (get bonded pairs) (patch_cluster is in column 6)
                    bonded_clusters.reset_index(drop=True, inplace=True)
                    for i in range(len(bonded_clusters)-1):
                        id1, id2 = int(bonded_clusters.iloc[i, 0]), int(bonded_clusters.iloc[i+1, 0]) # get ids of current and next cluster
                        if bonded_clusters.iloc[i, 6] == bonded_clusters.iloc[i+1, 6]: # if same patch_cluster, they are bonded
                            current_bonds.add(id1) # add to current bonds set
                            current_bonds.add(id2) # add to current bonds set
                    # clusters = np.zeros((N_patches,7)) # 7 values per line: id, mol, x, y, z, patch_coord, patch_cluster
                    # for i in range(N_patches):
                    #     clusters[i] = np.array(f_off.readline().strip().split(), dtype=float)
                    # # filter only consider bonds with at least 1 patch
                    # clusters = clusters[clusters[:,5] > 0] # patch_coord is in column 5
                    # # sort by patch_cluster (get bonded pairs) 
                    # clusters = clusters[clusters[:,6].argsort()] # patch_cluster is in column 6
                    # for i in range(len(clusters)-1):
                    #     id1, id2 = int(clusters[i][0]), int(clusters[i+1][0]) # get ids of current and next cluster
                    #     if clusters[i][6] == clusters[i+1][6]: # if same patch_cluster, they are bonded
                    #         current_bonds.add(id1) # add to current bonds set
                    #         current_bonds.add(id2) # add to current bonds set

                    broken_bonds = all_ids - set(unbonded_ids.keys()) - current_bonds
                    # print(f"Broken bonds: {len(broken_bonds)}")
                    for id in broken_bonds: # just broken, set start of offtime and track timers
                        unbonded_ids[int(id)] = time
                    del broken_bonds # free memory
                    del current_bonds # free memory

                    # print(f"Current unbonded_ids: {len(unbonded_ids)}, Total bond offtimes: {len(bond_offtimes)}")
                    # print(bond_lifetimes)
                    pbar.update(10 * timestep)
                
    return bond_offtimes



<>:77: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:123: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:77: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:123: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/tmp/ipykernel_431460/1629577861.py:77: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  clusters = pd.read_csv(f_on, delimiter='\s+', header=None, nrows=N_patches, names=['id', 'mol', 'x', 'y', 'z', 'patch_coord', 'patch_cluster'])
/tmp/ipykernel_431460/1629577861.py:123: SyntaxWa

In [31]:
from joblib import Parallel, delayed
from time import time

# cutoffs = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
cutoffs = [0.4, 0.5, 0.6, 0.7]

As = [80, 110]

def process_and_save_offtimes(cutoff_on, cutoff_off, A):
    start = time()
    filename_on = f"data/patches/{cutoff_on}prodpatch_chainlength_31_patchspacing_5_N_beads_24800_gaussA_{A}_r0_0.25_gaussB_10_seed_12345.bin.txt"
    filename_off = f"data/patches/{cutoff_off}prodpatch_chainlength_31_patchspacing_5_N_beads_24800_gaussA_{A}_r0_0.25_gaussB_10_seed_12345.bin.txt"
    offtimes = process_buffered_offtimes(filename_on, filename_off)
    offtimes_df = pd.DataFrame(offtimes, columns=["id", "start_time", "end_time", "offtime"])
    offtimes_df.to_csv(f"data/offtimes/buffered_offtimes_A{A}_cutoff_on{cutoff_on}_cutoff_off{cutoff_off}_r0_0.25.csv", index=False)
    print(f"Processed offtimes for A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}, total offtimes recorded: {len(offtimes)} in {time() - start:.2f} seconds")

# process_and_save_offtimes(0.7, 0.7, 100)

# only process offtimes for cutoff_on < cutoff_off to avoid duplicates
finish = Parallel(n_jobs=-1)(delayed(process_and_save_offtimes)(cutoff_on, cutoff_off, A) for cutoff_on in cutoffs for cutoff_off in cutoffs if cutoff_on < cutoff_off for A in As)



FileNotFoundError: [Errno 2] No such file or directory: 'data/patches/0.4prodpatch_chainlength_31_patchspacing_5_N_beads_24800_gaussA_80_r0_0.25_gaussB_10_seed_12345.bin.txt'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# for cutoff in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
#     fig = plt.figure(figsize=(8, 6))
#     for A in [50, 60, 70, 80, 90, 100]:
#         df = pd.read_csv(f"data/lifetimes/lifetimes_A{A}_cutoff{cutoff}_r0_0.25.csv")
#         lifetimes = np.array(df["lifetime"].values)
#         # print(f"Processed lifetimes for A={A}, cutoff={cutoff}, total lifetimes recorded: {len(lifetimes)}")

#         values, counts = np.unique(lifetimes, return_counts=True)


#         plt.hist(values, weights=counts, bins=50, alpha=0.5, label=f"A={A}", density=True)
#     plt.xlabel("Lifetime (LJ units)")
#     plt.ylabel("Counts")
#     plt.grid()
#     plt.title(f"Distribution of Bond Lifetimes for cutoff={cutoff}")
#     plt.legend()
#     plt.show()

As = [80]

for cutoff_off in [0.5, 0.6, 0.7]:
    fig = plt.figure(figsize=(8, 6))
    for cutoff_on in [0.4, 0.5, 0.6, 0.7]:
        if cutoff_on < cutoff_off:
            for A in As:
                df = pd.read_csv(f"data/offtimes/buffered_offtimes_A{A}_cutoff_on{cutoff_on}_cutoff_off{cutoff_off}_r0_0.25.csv")
                offtimes = np.array(df["offtime"].values)
                # print(f"Processed lifetimes for A={A}, cutoff={cutoff}, total lifetimes recorded: {len(lifetimes)}")

                # values, counts = np.unique(lifetimes, return_counts=True)
                hist, bins = np.histogram(offtimes, bins=50)


                plt.plot(bins[:-1], hist, alpha=0.5, label=f"A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}")
                print(f"Processed offtimes for A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}, total offtimes recorded: {len(offtimes)}")
        # if cutoff_on == cutoff_off:
        #     for A in As:
        #         df = pd.read_csv(f"data/offtimes/buffered_offtimes_A{A}_cutoff_on{cutoff_on}_cutoff_off{cutoff_off}_r0_0.25.csv")
        #         offtimes = np.array(df["offtime"].values)
        #         # print(f"Processed lifetimes for A={A}, cutoff={cutoff}, total lifetimes recorded: {len(lifetimes)}")

        #         # values, counts = np.unique(lifetimes, return_counts=True)
        #         hist, bins = np.histogram(offtimes, bins=50)


        #         plt.plot(bins[:-1], hist, alpha=0.5, label=f"A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}")
        #         print(f"Processed offtimes for A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}, total offtimes recorded: {len(offtimes)}")
    plt.xlabel("Lifetime (LJ units)")
    plt.ylabel("Density")
    plt.yscale("log")
    # plt.ylim(0, 0.2)
    plt.grid()
    plt.title(r"Distribution of Bond Offtimes for different $r_{\text{off}}, r_{\text{on}}$ for A=100")
    plt.legend()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


def biexponential(x, a1, b1, a2, b2):
    return a1 * np.exp(-b1 * x) + a2 * np.exp(-b2 * x)

def single_exponential(x, a, b):
    return a * np.exp(-b * x)




for cutoff_off in [ 0.5, 0.6, 0.7]:
    fig = plt.figure(figsize=(8, 6))
    for cutoff_on in [0.4, 0.5, 0.6, 0.7]:
        if cutoff_on < cutoff_off:
            for A in [130]:
                df = pd.read_csv(f"data/lifetimes/buffered_lifetimes_A{A}_cutoff_on{cutoff_on}_cutoff_off{cutoff_off}_r0_0.25.csv")
                lifetimes = np.array(df["lifetime"].values)
                # print(f"Processed lifetimes for A={A}, cutoff={cutoff}, total lifetimes recorded: {len(lifetimes)}")

                # values, counts = np.unique(lifetimes, return_counts=True)
                hist, bins = np.histogram(lifetimes, bins=50)
                print(f"Processed lifetimes for A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}, total lifetimes recorded: {len(lifetimes)}")
                hist = hist[1:] # remove first bin
                bins = bins[1:] # remove first bin
                print(f"Removing, remaining lifetimes recorded: {np.sum(hist)}")

                coeffs, cov = curve_fit(biexponential, bins[:-1], hist, p0=[1, 0.01, 1, 0.01], maxfev=10000)
                print(f"Fitted coefficients for A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}: {coeffs}")
                tau_1, tau_2 = sorted((1/coeffs[1], 1/coeffs[3]))
                print(f"Fitted lifetimes for A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}: tau_1={tau_1:.2f}, tau_2={tau_2:.2f}")

                plt.plot(bins[:-1], hist, alpha=0.5, label=f"A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}")
                plt.plot(bins[:-1], biexponential(bins[:-1], *coeffs), '--', label=r"Fit $\tau_1=$" + f"{tau_1:.2f}" + r", $\tau_2=$" + f"{tau_2:.2f}")

                # coeffs, cov = curve_fit(single_exponential, bins[:-1], hist, p0=[1, 0.01])
                # print(f"Fitted coefficients for A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}: {coeffs}")
                # plt.plot(bins[:-1], hist, alpha=0.5, label=f"A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}")
                # plt.plot(bins[:-1], single_exponential(bins[:-1], *coeffs), '--', label=f"Fit A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}")
            
            # plt.xlabel("Lifetime (LJ units)")
            # plt.ylabel("Density")
            # # plt.yscale("log")
            # # plt.ylim(0, 0.2)
            # plt.grid()
            # plt.title(f"Distribution of Bond Lifetimes for A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}")
            # plt.legend()
            # plt.show()
            
        if cutoff_on == cutoff_off:
            for A in [130]:
                df = pd.read_csv(f"data/lifetimes/lifetimes_A{A}_cutoff{cutoff_on}_r0_0.25.csv")
                lifetimes = np.array(df["lifetime"].values)
                # print(f"Processed lifetimes for A={A}, cutoff={cutoff}, total lifetimes recorded: {len(lifetimes)}")

                # values, counts = np.unique(lifetimes, return_counts=True)
                hist, bins = np.histogram(lifetimes, bins=50)


                print(f"Processed lifetimes for A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}, total lifetimes recorded: {len(lifetimes)}")
                hist = hist[1:] # remove first bin
                bins = bins[1:] # remove first bin
                print(f"Removing, remaining lifetimes recorded: {np.sum(hist)}")

                coeffs, cov = curve_fit(biexponential, bins[:-1], hist, p0=[1, 0.01, 1, 0.01], maxfev=10000)
                print(f"Fitted coefficients for A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}: {coeffs}")
                tau_1, tau_2 = sorted((1/coeffs[1], 1/coeffs[3]))
                print(f"Fitted lifetimes for A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}: tau_1={tau_1:.2f}, tau_2={tau_2:.2f}")

                
                plt.plot(bins[:-1], hist, alpha=0.5, label=f"A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}")
                plt.plot(bins[:-1], biexponential(bins[:-1], *coeffs), '--', label=r"Fit $\tau_1=$" + f"{tau_1:.2f}" + r", $\tau_2=$" + f"{tau_2:.2f}")

                # coeffs, cov = curve_fit(single_exponential, bins[:-1], hist, p0=[1, 0.01])
                # print(f"Fitted coefficients for A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}: {coeffs}")
                # plt.plot(bins[:-1], hist, alpha=0.5, label=f"A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}")
                # plt.plot(bins[:-1], single_exponential(bins[:-1], *coeffs), '--', label=f"Fit A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}")
                
            # plt.xlabel("Lifetime (LJ units)")
            # plt.ylabel("Density")
            # # plt.yscale("log")
            # # plt.ylim(0, 0.2)
            # plt.grid()
            # plt.title(f"Distribution of Bond Lifetimes for A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}")
            # plt.legend()
            # plt.show()
    plt.xlabel("Lifetime (LJ units)")
    plt.ylabel("Density")
    plt.yscale("log")
    # plt.ylim(0, 0.2)
    plt.grid()
    plt.title(f"Distribution of Bond Lifetimes Assorted r_on, r_off={cutoff_off} for A={A}")
    plt.legend()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# for cutoff in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
#     fig = plt.figure(figsize=(8, 6))
#     for A in [50, 60, 70, 80, 90, 100]:
#         df = pd.read_csv(f"data/lifetimes/lifetimes_A{A}_cutoff{cutoff}_r0_0.25.csv")
#         lifetimes = np.array(df["lifetime"].values)
#         # print(f"Processed lifetimes for A={A}, cutoff={cutoff}, total lifetimes recorded: {len(lifetimes)}")

#         values, counts = np.unique(lifetimes, return_counts=True)


#         plt.hist(values, weights=counts, bins=50, alpha=0.5, label=f"A={A}", density=True)
#     plt.xlabel("Lifetime (LJ units)")
#     plt.ylabel("Counts")
#     plt.grid()
#     plt.title(f"Distribution of Bond Lifetimes for cutoff={cutoff}")
#     plt.legend()
#     plt.show()
fig = plt.figure(figsize=(8, 6))
for cutoff_on in [0.4, 0.5, 0.6]:
    
    for cutoff_off in [0.4, 0.5, 0.6, 0.7]:
        if cutoff_on < cutoff_off:
            for A in [100]:
                df = pd.read_csv(f"data/lifetimes/buffered_lifetimes_A{A}_cutoff_on{cutoff_on}_cutoff_off{cutoff_off}_r0_0.25.csv")
                lifetimes = np.array(df["lifetime"].values)
                # print(f"Processed lifetimes for A={A}, cutoff={cutoff}, total lifetimes recorded: {len(lifetimes)}")

                # values, counts = np.unique(lifetimes, return_counts=True)


                plt.hist(lifetimes, bins=50, alpha=0.5,density=True, label=f"A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}")
                print(f"Processed lifetimes for A={A}, cutoff_on={cutoff_on}, cutoff_off={cutoff_off}, total lifetimes recorded: {len(lifetimes)}")
plt.xlabel("Lifetime (LJ units)")
plt.ylabel("Density")
plt.yscale("log")
# plt.ylim(0, 0.2)
plt.grid()
plt.title(f"Distribution of Bond Lifetimes for cutoff_on={cutoff_on}")
plt.legend()
plt.show()